In [1]:
# !pip install -q ultralytics ensemble-boxes albumentations

# import os, yaml, random, shutil, numpy as np, pandas as pd, torch, cv2, json, gc
# from ultralytics import YOLO
# from sklearn.model_selection import train_test_split, KFold
# from tqdm import tqdm
# from ensemble_boxes import weighted_boxes_fusion
# import warnings
# warnings.filterwarnings('ignore')

# # Настройка для CUDA с оптимизацией памяти
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f'Using device: {device}')
# if device == 'cuda':
#     print(f'GPU: {torch.cuda.get_device_name(0)}')
#     print(f'Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    
#     # Оптимизации памяти
#     torch.cuda.empty_cache()
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cuda.matmul.allow_tf32 = True
#     torch.backends.cudnn.allow_tf32 = True
    
#     # Устанавливаем лимит памяти (90% от доступной)
#     total_memory = torch.cuda.get_device_properties(0).total_memory
#     torch.cuda.set_per_process_memory_fraction(0.9)

# def set_seed(seed=42):
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
# set_seed(42)

# base_path = '/kaggle/input/competitions/dl-lab-2-stuff-detection'
# yolo_base = f'{base_path}/yolo_dataset/yolo_dataset'
# train_img_path = f'{yolo_base}/train/images'
# train_lbl_path = f'{yolo_base}/train/labels'
# test_path = f'{base_path}/test_images/test_images'
# sample_sub = pd.read_csv(f'{base_path}/sample_sub.csv')

# # ==================== ПОДГОТОВКА ДАННЫХ ====================
# print("\n=== Подготовка данных ===")
# imgs = os.listdir(train_img_path)
# by_type = {'both':[], 'cust':[], 'staff':[], 'none':[]}

# for img in tqdm(imgs, desc="Анализ изображений"):
#     lp = os.path.join(train_lbl_path, img.replace('.jpg','.txt'))
#     c, s = False, False
#     if os.path.exists(lp):
#         with open(lp) as f:
#             for line in f:
#                 cls = int(line.strip().split()[0])
#                 if cls==0: c=True
#                 elif cls==1: s=True
#     if c and s: by_type['both'].append(img)
#     elif c: by_type['cust'].append(img)
#     elif s: by_type['staff'].append(img)
#     else: by_type['none'].append(img)

# print(f"Статистика датасета:")
# print(f"  both: {len(by_type['both'])}")
# print(f"  customer: {len(by_type['cust'])}")
# print(f"  staff: {len(by_type['staff'])}")
# print(f"  none: {len(by_type['none'])}")

# # ==================== ОПТИМИЗИРОВАННЫЙ ДЕТЕКТОР ====================
# class OptimizedStaffDetector:
#     def __init__(self, n_folds=2):  # Уменьшили до 2 фолдов
#         self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
#         self.n_folds = n_folds
#         self.models = []
#         self.fold_datasets = []
#         self.test_tta_transforms = self.get_tta_transforms()
        
#     def get_tta_transforms(self):
#         """Test Time Augmentation transforms"""
#         return [
#             ('original', lambda x: x),
#             ('h_flip', lambda x: cv2.flip(x, 1)),
#         ]
    
#     def prepare_fold_data(self, train_imgs, val_imgs, fold):
#         """Подготовка данных для конкретного фолда (без копирования)"""
#         fold_dir = f'/kaggle/working/dataset_fold_{fold}'
        
#         # Создание структуры директорий
#         for split in ['train', 'val']:
#             os.makedirs(f'{fold_dir}/{split}/images', exist_ok=True)
#             os.makedirs(f'{fold_dir}/{split}/labels', exist_ok=True)
        
#         # Создаем символические ссылки вместо копирования
#         for split, imgs_list in [('train', train_imgs), ('val', val_imgs)]:
#             for img in tqdm(imgs_list, desc=f'Fold {fold} - линковка {split}'):
#                 # Ссылка на изображение
#                 src_img = os.path.join(train_img_path, img)
#                 dst_img = os.path.join(fold_dir, split, 'images', img)
#                 if not os.path.exists(dst_img):
#                     os.symlink(src_img, dst_img)
                
#                 # Ссылка на метку
#                 lbl = img.replace('.jpg', '.txt')
#                 src_lbl = os.path.join(train_lbl_path, lbl)
#                 dst_lbl = os.path.join(fold_dir, split, 'labels', lbl)
#                 if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
#                     os.symlink(src_lbl, dst_lbl)
        
#         # Создание data.yaml
#         with open(f'{fold_dir}/data.yaml','w') as f:
#             yaml.dump({
#                 'path': fold_dir,
#                 'train': 'train/images',
#                 'val': 'val/images',
#                 'nc': 2,
#                 'names': ['customer', 'staff']
#             }, f)
        
#         return fold_dir
    
#     def train_fold(self, train_imgs, val_imgs, fold):
#         """Обучение одной модели для фолда с оптимизацией памяти"""
#         print(f"\n{'='*50}")
#         print(f"Обучение Fold {fold+1}/{self.n_folds}")
#         print(f"{'='*50}")
        
#         # Подготовка данных
#         fold_dir = self.prepare_fold_data(train_imgs, val_imgs, fold)
#         self.fold_datasets.append(fold_dir)
        
#         # Очистка памяти
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.reset_peak_memory_stats()
        
#         # Загрузка модели - используем nano для экономии памяти
#         model = YOLO('yolov8n.pt')
        
#         # Обучение с оптимизацией памяти
#         results = model.train(
#             data=f'{fold_dir}/data.yaml',
#             epochs=40,
#             imgsz=480,
#             batch=8,
#             optimizer='AdamW',
#             device=self.device,
#             lr0=0.001,
#             lrf=0.01,
#             weight_decay=0.0005,
#             warmup_epochs=3,
            
#             # Аугментации
#             hsv_h=0.015,
#             hsv_s=0.7,
#             hsv_v=0.4,
#             degrees=2.0,
#             translate=0.1,
#             scale=0.5,
#             flipud=0.0,
#             fliplr=0.5,
#             mosaic=0.8,
            
#             # Оптимизация памяти
#             close_mosaic=8,
#             multi_scale=False,
#             amp=True,
#             cache=False,
#             workers=2,
#             patience=15,
            
#             # Сохранение
#             project=f'/kaggle/working/yolo_fold_{fold}',
#             name='train',
#             exist_ok=True,
#             verbose=True,
#             save=True,
            
#             # Валидация
#             val=True,
#             plots=False,
#             max_det=100,
#         )
        
#         # Загрузка лучшей модели
#         best_model_path = f'/kaggle/working/yolo_fold_{fold}/train/weights/best.pt'
#         if os.path.exists(best_model_path):
#             print(f"✅ Лучшая модель сохранена: {best_model_path}")
#             model = YOLO(best_model_path)
#         else:
#             print(f"⚠️ Использую последнюю модель")
#             model = YOLO(f'/kaggle/working/yolo_fold_{fold}/train/weights/last.pt')
        
#         self.models.append(model)
        
#         # Очистка после обучения
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
        
#         return model
    
#     def train_with_cross_validation(self):
#         """Обучение с кросс-валидацией"""
#         # Создание списка всех изображений
#         all_images = []
#         for k, v in by_type.items():
#             all_images.extend(v)
        
#         # Стратифицированное разбиение
#         kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
        
#         # Для стратификации создаем метки классов
#         y_labels = []
#         for img in all_images:
#             if img in by_type['both']:
#                 y_labels.append(0)
#             elif img in by_type['cust']:
#                 y_labels.append(1)
#             elif img in by_type['staff']:
#                 y_labels.append(2)
#             else:
#                 y_labels.append(3)
        
#         for fold, (train_idx, val_idx) in enumerate(kf.split(all_images, y_labels)):
#             train_imgs = [all_images[i] for i in train_idx]
#             val_imgs = [all_images[i] for i in val_idx]
            
#             self.train_fold(train_imgs, val_imgs, fold)
            
#             # Очистка памяти после каждого фолда
#             gc.collect()
#             if torch.cuda.is_available():
#                 torch.cuda.empty_cache()
    
#     def predict_with_ensemble(self, test_images, conf_threshold=0.25, iou_threshold=0.5):
#         """Ансамблевое предсказание с TTA"""
#         all_predictions = []
        
#         for img_path in tqdm(test_images, desc="Ансамблевое предсказание"):
#             img = cv2.imread(img_path)
#             if img is None:
#                 all_predictions.append([])
#                 continue
            
#             h, w = img.shape[:2]
#             all_boxes = []
#             all_scores = []
            
#             # Для каждой модели в ансамбле
#             for model_idx, model in enumerate(self.models):
#                 model_boxes = []
#                 model_scores = []
                
#                 # TTA для каждой модели
#                 for tta_name, tta_transform in self.test_tta_transforms:
#                     # Применяем TTA
#                     img_tta = tta_transform(img.copy())
                    
#                     # Предсказание
#                     results = model.predict(
#                         img_tta,
#                         conf=conf_threshold,
#                         iou=iou_threshold,
#                         classes=[1],
#                         verbose=False,
#                         device=self.device,
#                         imgsz=480,
#                         max_det=50,
#                         amp=True,
#                     )[0]
                    
#                     if results.boxes is not None and len(results.boxes) > 0:
#                         boxes = results.boxes.xyxy.cpu().numpy()
#                         scores = results.boxes.conf.cpu().numpy()
                        
#                         # Инвертируем трансформации для координат
#                         if tta_name == 'h_flip':
#                             boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                        
#                         model_boxes.extend(boxes)
#                         model_scores.extend(scores)
                    
#                     # Очистка после каждой TTA
#                     del results
#                     if torch.cuda.is_available():
#                         torch.cuda.empty_cache()
                
#                 # WBF для объединения TTA предсказаний одной модели
#                 if model_boxes:
#                     # Нормализация координат для WBF
#                     norm_boxes = np.array(model_boxes).copy()
#                     norm_boxes[:, [0, 2]] /= w
#                     norm_boxes[:, [1, 3]] /= h
                    
#                     try:
#                         fused_boxes, fused_scores, _ = weighted_boxes_fusion(
#                             [norm_boxes.tolist()],
#                             [model_scores],
#                             [np.ones(len(model_boxes)).tolist()],
#                             iou_thr=0.5,
#                             skip_box_thr=conf_threshold
#                         )
                        
#                         # Денормализация
#                         if len(fused_boxes) > 0:
#                             fused_boxes = np.array(fused_boxes)
#                             fused_boxes[:, [0, 2]] *= w
#                             fused_boxes[:, [1, 3]] *= h
                            
#                             all_boxes.extend(fused_boxes)
#                             all_scores.extend(fused_scores)
#                     except:
#                         all_boxes.extend(model_boxes)
#                         all_scores.extend(model_scores)
            
#             # Финальное форматирование с конвертацией в float
#             if all_boxes:
#                 formatted_boxes = []
#                 for box, score in zip(all_boxes, all_scores):
#                     x1, y1, x2, y2 = box
#                     cx = float((x1 + x2) / 2 / w)  # Конвертируем в float
#                     cy = float((y1 + y2) / 2 / h)  # Конвертируем в float
#                     bw = float((x2 - x1) / w)      # Конвертируем в float
#                     bh = float((y2 - y1) / h)      # Конвертируем в float
#                     score = float(score)            # Конвертируем в float
                    
#                     # Простая фильтрация
#                     if bw * bh > 0.001 and bw < 0.9 and bh < 0.9:
#                         formatted_boxes.append([cx, cy, bw, bh, score])
                
#                 all_predictions.append(formatted_boxes)
#             else:
#                 all_predictions.append([])
            
#             # Периодическая очистка
#             if len(all_predictions) % 50 == 0:
#                 gc.collect()
#                 if torch.cuda.is_available():
#                     torch.cuda.empty_cache()
        
#         return all_predictions
    
#     def save_models_info(self):
#         """Сохранение информации о моделях"""
#         info = {
#             'n_folds': self.n_folds,
#             'models': []
#         }
        
#         for i, model in enumerate(self.models):
#             model_path = f'/kaggle/working/yolo_fold_{i}/train/weights/best.pt'
#             if os.path.exists(model_path):
#                 info['models'].append({
#                     'fold': i,
#                     'path': model_path,
#                     'size_mb': float(os.path.getsize(model_path) / 1024**2)  # Конвертируем в float
#                 })
        
#         with open('/kaggle/working/ensemble_info.json', 'w') as f:
#             json.dump(info, f, indent=2)
        
#         print(f"\n=== Информация об ансамбле ===")
#         print(f"Количество моделей: {len(info['models'])}")
#         for m in info['models']:
#             print(f"  Fold {m['fold']}: {m['size_mb']:.2f} MB")

# # ==================== ОСНОВНОЙ ПРОЦЕСС ====================
# def main():
#     print("\n=== Запуск оптимизированного детектора с фолдами ===")
    
#     # Создание детектора
#     detector = OptimizedStaffDetector(n_folds=2)
    
#     # Обучение с кросс-валидацией
#     print("\n=== Обучение ансамбля моделей ===")
#     detector.train_with_cross_validation()
    
#     # Сохранение информации о моделях
#     detector.save_models_info()
    
#     # Проверка сохраненных моделей
#     print("\n=== Проверка сохраненных моделей ===")
#     for i in range(detector.n_folds):
#         best_path = f'/kaggle/working/yolo_fold_{i}/train/weights/best.pt'
#         last_path = f'/kaggle/working/yolo_fold_{i}/train/weights/last.pt'
        
#         if os.path.exists(best_path):
#             size_mb = os.path.getsize(best_path) / 1024**2
#             print(f"✅ Fold {i}: best.pt - {size_mb:.2f} MB")
#         elif os.path.exists(last_path):
#             size_mb = os.path.getsize(last_path) / 1024**2
#             print(f"⚠️ Fold {i}: last.pt - {size_mb:.2f} MB")
#         else:
#             print(f"❌ Fold {i}: модели не найдены!")
    
#     # Инференс на тестовых данных
#     print("\n=== Инференс на тестовых данных ===")
#     test_images = []
#     for img_name in sample_sub['image_name']:
#         img_path = os.path.join(test_path, img_name)
#         if os.path.exists(img_path):
#             test_images.append(img_path)
    
#     # Получение предсказаний
#     predictions = detector.predict_with_ensemble(
#         test_images,
#         conf_threshold=0.2,
#         iou_threshold=0.5
#     )
    
#     # Форматирование для сабмишна
#     print("\n=== Форматирование результатов ===")
#     formatted_preds = []
    
#     for pred in predictions:
#         if pred:
#             # Сортировка по уверенности
#             pred.sort(key=lambda x: x[4], reverse=True)
#             # Ограничение количества
#             pred = pred[:30]
            
#             # Дополнительная проверка на типы данных
#             clean_pred = []
#             for p in pred:
#                 # Убеждаемся, что все значения - обычные float
#                 clean_pred.append([float(x) for x in p])
            
#             formatted_preds.append(json.dumps(clean_pred, separators=(',',':')))
#         else:
#             formatted_preds.append('[]')
    
#     # Создание сабмишна
#     sub = pd.DataFrame({
#         'id': sample_sub['id'],
#         'image_name': sample_sub['image_name'],
#         'boxes': formatted_preds
#     })
    
#     # Сохранение
#     sub.to_csv('/kaggle/working/submission_ensemble.csv', index=False)
    
#     print(f"\n✅ Сабмишн сохранен: /kaggle/working/submission_ensemble.csv")
#     print(f"   Всего изображений: {len(sub)}")
#     print(f"   Изображений с детекциями: {sum(1 for p in formatted_preds if p != '[]')}")
#     print(f"   Процент детекций: {sum(1 for p in formatted_preds if p != '[]')/len(formatted_preds)*100:.2f}%")
    
#     # Статистика по предсказаниям
#     boxes_per_image = [len(json.loads(p)) for p in formatted_preds if p != '[]']
#     if boxes_per_image:
#         print(f"   Среднее боксов на изображение: {np.mean(boxes_per_image):.2f}")
#         print(f"   Макс боксов: {max(boxes_per_image)}")
    
#     # Финальный отчет по памяти
#     if device == 'cuda':
#         print(f"\n=== Memory Usage Report ===")
#         print(f"Final VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
#         print(f"Peak VRAM usage: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
    
#     return predictions

# # ==================== ЗАПУСК ====================
# if __name__ == "__main__":
#     predictions = main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.0 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
GPU: Tesla P100-PCIE-16GB
Available VRAM: 15.9 GB

=== Подготовка данных ===


Анализ изображений: 100%|██████████| 3908/3908 [00:31<00:00, 123.22it/s]


Статистика датасета:
  both: 2662
  customer: 819
  staff: 427
  none: 0

=== Запуск оптимизированного детектора с фолдами ===

=== Обучение ансамбля моделей ===

Обучение Fold 1/2


Fold 0 - линковка val: 100%|██████████| 1954/1954 [00:01<00:00, 1338.45it/s]


Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=8, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_fold_0/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=100, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.8, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patie

Fold 1 - линковка val: 100%|██████████| 1954/1954 [00:02<00:00, 667.70it/s]


Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=8, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_fold_1/data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=100, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.8, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patie

Ансамблевое предсказание: 100%|██████████| 4454/4454 [06:24<00:00, 11.57it/s]


=== Форматирование результатов ===

✅ Сабмишн сохранен: /kaggle/working/submission_ensemble.csv
   Всего изображений: 4454
   Изображений с детекциями: 1966
   Процент детекций: 44.14%
   Среднее боксов на изображение: 2.27
   Макс боксов: 9

=== Memory Usage Report ===
Final VRAM usage: 0.05 GB
Peak VRAM usage: 1.26 GB


In [2]:

!pip install -q ultralytics ensemble-boxes seaborn


import os
import sys
import yaml
import random
import shutil
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
import json


class Config:
    """Централизованная конфигурация"""
    SEED = 42
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    N_FOLDS = 3
    BASE_PATH = '/kaggle/input/competitions/dl-lab-2-stuff-detection'
    YOLO_BASE = f'{BASE_PATH}/yolo_dataset/yolo_dataset'
    TRAIN_IMAGES = f'{YOLO_BASE}/train/images'
    TRAIN_LABELS = f'{YOLO_BASE}/train/labels'
    TEST_PATH = f'{BASE_PATH}/test_images/test_images'
    SAMPLE_SUB = f'{BASE_PATH}/sample_sub.csv'
    WORK_DIR = '/kaggle/working'

config = Config()


def seed_everything(seed_value):
    """Фиксация всех random seed"""
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"✅ SEED УСТАНОВЛЕН: {seed_value}")

seed_everything(config.SEED)


print("="*70)
print("ЭТАП 1: ЗАГРУЗКА И АНАЛИЗ ДАННЫХ")
print("="*70)

def load_and_analyze():
    """Загрузка и первичный анализ датасета"""
    train_images = os.listdir(config.TRAIN_IMAGES)
    print(f"Найдено изображений: {len(train_images)}")
    
    # Анализ меток
    label_stats = {
        'with_customer': 0,
        'with_staff': 0,
        'with_both': 0,
        'empty': 0,
        'total_boxes': 0,
        'customer_boxes': 0,
        'staff_boxes': 0,
        'invalid': 0
    }
    
    box_sizes = []
    
    for img_name in tqdm(train_images, desc="Анализ меток"):
        label_path = os.path.join(config.TRAIN_LABELS, img_name.replace('.jpg', '.txt'))
        has_customer = False
        has_staff = False
        
        if os.path.exists(label_path):
            try:
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cls_id = int(parts[0])
                            w = float(parts[3])
                            h = float(parts[4])
                            box_sizes.append(w * h)
                            
                            if cls_id == 0:
                                has_customer = True
                                label_stats['customer_boxes'] += 1
                            elif cls_id == 1:
                                has_staff = True
                                label_stats['staff_boxes'] += 1
                            else:
                                label_stats['invalid'] += 1
                        label_stats['total_boxes'] += len(lines)
            except Exception as e:
                print(f"Ошибка чтения {label_path}: {e}")
        
        if has_customer and has_staff:
            label_stats['with_both'] += 1
        elif has_customer:
            label_stats['with_customer'] += 1
        elif has_staff:
            label_stats['with_staff'] += 1
        else:
            label_stats['empty'] += 1
    
    return train_images, label_stats, box_sizes

train_images_list, stats, sizes = load_and_analyze()

print("\nРЕЗУЛЬТАТЫ АНАЛИЗА:")
print(f"  Изображения с обоими классами: {stats['with_both']:5d}")
print(f"  Только customer: {stats['with_customer']:5d}")
print(f"  Только staff:    {stats['with_staff']:5d}")
print(f"  Пустые:          {stats['empty']:5d}")
print(f"  Всего боксов:    {stats['total_boxes']:5d}")
print(f"  Customer боксов: {stats['customer_boxes']:5d}")
print(f"  Staff боксов:    {stats['staff_boxes']:5d}")



print("\n" + "="*70)
print("ЭТАП 2: ПОДГОТОВКА 3-ФОЛД КРОСС-ВАЛИДАЦИИ")
print("="*70)

def prepare_cross_validation_folds(images, n_folds=3):
    """Создание стратифицированных фолдов"""
    # Создаем метки для стратификации
    labels_for_strat = []
    image_names = []
    
    for img_name in tqdm(images, desc="Создание страт"):
        label_path = os.path.join(config.TRAIN_LABELS, img_name.replace('.jpg', '.txt'))
        label_type = 'empty'
        
        if os.path.exists(label_path):
            has_customer = False
            has_staff = False
            with open(label_path, 'r') as f:
                for line in f:
                    cls_id = int(line.strip().split()[0])
                    if cls_id == 0:
                        has_customer = True
                    elif cls_id == 1:
                        has_staff = True
            if has_customer and has_staff:
                label_type = 'both'
            elif has_customer:
                label_type = 'customer'
            elif has_staff:
                label_type = 'staff'
        
        labels_for_strat.append(label_type)
        image_names.append(img_name)
    
    # Преобразуем в числовые метки
    type_to_num = {'both': 0, 'customer': 1, 'staff': 2, 'empty': 3}
    strat_labels = [type_to_num[t] for t in labels_for_strat]
    
    # Создаем фолды
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=config.SEED)
    folds = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(image_names, strat_labels)):
        train_fold = [image_names[i] for i in train_idx]
        val_fold = [image_names[i] for i in val_idx]
        folds.append({
            'fold': fold_idx,
            'train': train_fold,
            'val': val_fold
        })
        print(f"  Fold {fold_idx}: train={len(train_fold)}, val={len(val_fold)}")
    
    return folds, image_names, strat_labels

folds_data, all_images, strat_labels = prepare_cross_validation_folds(train_images_list, config.N_FOLDS)


def create_fold_dataset(fold_info, base_dir):
    """Создание структуры датасета для конкретного фолда"""
    fold_dir = f'{config.WORK_DIR}/fold_{fold_info["fold"]}_dataset'
    
    # Создаем директории
    for split in ['train', 'val']:
        os.makedirs(f'{fold_dir}/{split}/images', exist_ok=True)
        os.makedirs(f'{fold_dir}/{split}/labels', exist_ok=True)
    
    # Копируем файлы (используем hard links если возможно)
    for split_name, image_list in [('train', fold_info['train']), ('val', fold_info['val'])]:
        for img_name in tqdm(image_list, desc=f"Fold {fold_info['fold']} - {split_name}"):
            # Копируем изображение
            src_img = os.path.join(config.TRAIN_IMAGES, img_name)
            dst_img = os.path.join(fold_dir, split_name, 'images', img_name)
            try:
                os.link(src_img, dst_img)
            except:
                shutil.copy2(src_img, dst_img)
            
            # Копируем метку
            label_name = img_name.replace('.jpg', '.txt')
            src_lbl = os.path.join(config.TRAIN_LABELS, label_name)
            dst_lbl = os.path.join(fold_dir, split_name, 'labels', label_name)
            if os.path.exists(src_lbl):
                try:
                    os.link(src_lbl, dst_lbl)
                except:
                    shutil.copy2(src_lbl, dst_lbl)
    
    # Создаем data.yaml
    yaml_content = {
        'path': fold_dir,
        'train': 'train/images',
        'val': 'val/images',
        'nc': 2,
        'names': ['customer', 'staff']
    }
    
    with open(f'{fold_dir}/data.yaml', 'w') as f:
        yaml.dump(yaml_content, f)
    
    print(f"Датасет для fold {fold_info['fold']} создан в {fold_dir}")
    return fold_dir


class FoldTrainer:
    """Тренер для обучения одной модели на фолде"""
    
    def __init__(self, fold_idx, device):
        self.fold_idx = fold_idx
        self.device = device
        self.model = None
        self.best_path = None
        
    def setup_model(self):
        """Инициализация модели"""
        self.model = YOLO('yolov8m.pt')
        print(f"  Модель YOLOv8m загружена")
        
    def train(self, data_yaml, fold_dir):
        """Обучение модели"""
        print(f"\n   СТАРТ ОБУЧЕНИЯ FOLD {self.fold_idx}")
        
        # Очистка памяти перед обучением
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        
        # Параметры обучения
        train_params = {
            'data': data_yaml,
            'epochs': 40,
            'imgsz': 640,
            'batch': 14,
            'optimizer': 'AdamW',
            'lr0': 0.0007,
            'lrf': 0.01,
            'weight_decay': 0.0005,
            'warmup_epochs': 3,
            'warmup_momentum': 0.8,
            'warmup_bias_lr': 0.1,
            
            # Аугментации
            'hsv_h': 0.02,
            'hsv_s': 0.7,
            'hsv_v': 0.4,
            'degrees': 10.0,
            'translate': 0.1,
            'scale': 0.5,
            'shear': 2.0,
            'perspective': 0.0005,
            'flipud': 0.3,
            'fliplr': 0.5,
            'mosaic': 1.0,
            'mixup': 0.2,
            'copy_paste': 0.2,
            
            # Оптимизация
            'close_mosaic': 10,
            'multi_scale': False,
            'amp': True,
            'workers': 4,
            'patience': 15,
            'save': True,
            'save_period': 10,
            'plots': True,
            'project': f'{config.WORK_DIR}/yolo_fold_{self.fold_idx}',
            'name': 'run',
            'exist_ok': True,
            'seed': config.SEED + self.fold_idx,
            'verbose': False,
        }
        
        # Запуск обучения
        results = self.model.train(**train_params)
        
        # Сохраняем путь к лучшей модели
        self.best_path = f'{config.WORK_DIR}/yolo_fold_{self.fold_idx}/run/weights/best.pt'
        if os.path.exists(self.best_path):
            print(f"  ✅ Лучшая модель сохранена: {self.best_path}")
            self.model = YOLO(self.best_path)
        else:
            print(f"   Лучшая модель не найдена, использую последнюю")
            self.best_path = f'{config.WORK_DIR}/yolo_fold_{self.fold_idx}/run/weights/last.pt'
            self.model = YOLO(self.best_path)
        
        return self.model


print("\n" + "="*70)
print("ЭТАП 3: ОБУЧЕНИЕ 3-х МОДЕЛЕЙ")
print("="*70)

trained_models = []
fold_paths = []

for fold_info in folds_data:
    print(f"\n ОБРАБОТКА FOLD {fold_info['fold']}")
    print("-" * 50)
    
    # Создаем датасет для фолда
    fold_dataset_dir = create_fold_dataset(fold_info, config.WORK_DIR)
    fold_paths.append(fold_dataset_dir)
    
    # Создаем и обучаем модель
    trainer = FoldTrainer(fold_info['fold'], config.DEVICE)
    trainer.setup_model()
    trained_model = trainer.train(f'{fold_dataset_dir}/data.yaml', fold_dataset_dir)
    
    trained_models.append({
        'fold': fold_info['fold'],
        'model': trained_model,
        'path': trainer.best_path,
        'train_size': len(fold_info['train']),
        'val_size': len(fold_info['val'])
    })
    
    print(f"  ✅ Fold {fold_info['fold']} завершен")

print(f"\n ОБУЧЕНИЕ ЗАВЕРШЕНО: {len(trained_models)} модели")


print("\n" + "="*70)
print("ЭТАП 4: ОПТИМИЗАЦИЯ ПОРОГОВ")
print("="*70)

def find_best_threshold(models, val_images, conf_range=[0.1, 0.15, 0.2, 0.25, 0.3, 0.35]):
    """Поиск оптимального порога уверенности"""
    results = {}
    
    for thresh in conf_range:
        total_detections = 0
        images_with_detections = 0
        
        for img_name in tqdm(val_images[:50], desc=f"Порог {thresh:.2f}", leave=False):
            img_path = os.path.join(config.TRAIN_IMAGES, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue
            
            all_boxes = []
            for m in models:
                res = m['model'].predict(img, conf=thresh, classes=[1], verbose=False)[0]
                if res.boxes is not None:
                    all_boxes.extend(res.boxes.xyxy.cpu().numpy())
            
            if len(all_boxes) > 0:
                images_with_detections += 1
                total_detections += len(all_boxes)
        
        avg_per_image = total_detections / max(1, images_with_detections)
        detection_rate = images_with_detections / 50
        
        results[thresh] = {
            'total': total_detections,
            'detection_rate': detection_rate,
            'avg_per_image': avg_per_image
        }
        
        print(f"  Порог {thresh:.2f}: {total_detections} детекций, rate={detection_rate:.2f}")
    
    # Выбираем порог с максимальным detection rate
    best_thresh = max(results, key=lambda x: results[x]['detection_rate'])
    print(f"\n✅ Оптимальный порог: {best_thresh:.2f}")
    
    return best_thresh, results

# Берем первые 50 валидационных изображений для теста
val_samples = folds_data[0]['val'][:50]
optimal_threshold, threshold_stats = find_best_threshold(trained_models, val_samples)


def ensemble_predict(models, image, conf_thresh=0.2, iou_thresh=0.5):
    """Ансамблевое предсказание с WBF"""
    h, w = image.shape[:2]
    all_boxes = []
    all_scores = []
    
    for model_info in models:
        results = model_info['model'].predict(
            image, 
            conf=conf_thresh, 
            classes=[1],
            verbose=False,
            max_det=100
        )[0]
        
        if results.boxes is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            scores = results.boxes.conf.cpu().numpy()
            
            # Нормализация для WBF
            norm_boxes = boxes.copy()
            norm_boxes[:, [0, 2]] /= w
            norm_boxes[:, [1, 3]] /= h
            
            all_boxes.append(norm_boxes.tolist())
            all_scores.append(scores.tolist())
    
    if not all_boxes:
        return []
    
    # WBF объединение
    try:
        fused_boxes, fused_scores, _ = weighted_boxes_fusion(
            all_boxes,
            all_scores,
            [np.ones(len(s)) for s in all_scores],
            iou_thr=iou_thresh,
            skip_box_thr=conf_thresh
        )
        
        if len(fused_boxes) == 0:
            return []
        
        # Денормализация и форматирование
        fused_boxes = np.array(fused_boxes)
        fused_boxes[:, [0, 2]] *= w
        fused_boxes[:, [1, 3]] *= h
        
        results = []
        for box, score in zip(fused_boxes, fused_scores):
            x1, y1, x2, y2 = box
            cx = float(((x1 + x2) / 2) / w)
            cy = float(((y1 + y2) / 2) / h)
            width = float((x2 - x1) / w)
            height = float((y2 - y1) / h)
            
            if width * height > 0.001:
                results.append([cx, cy, width, height, float(score)])
        
        return results
    
    except Exception as e:
        print(f" Ошибка WBF: {e}")
        return []

# ===============================
# 11. ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ
# ===============================
print("\n" + "="*70)
print("ЭТАП 5: ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ")
print("="*70)

# Загрузка sample submission
sample_df = pd.read_csv(config.SAMPLE_SUB)
test_image_names = sample_df['image_name'].tolist()

# Прогнозирование
ensemble_predictions = []
staff_counts = []

for img_name in tqdm(test_image_names, desc="Обработка тестовых изображений"):
    img_path = os.path.join(config.TEST_PATH, img_name)
    
    if not os.path.exists(img_path):
        ensemble_predictions.append([])
        continue
    
    img = cv2.imread(img_path)
    if img is None:
        ensemble_predictions.append([])
        continue
    
    boxes = ensemble_predict(
        trained_models, 
        img, 
        conf_thresh=optimal_threshold,
        iou_thresh=0.5
    )
    
    ensemble_predictions.append(boxes)
    staff_counts.append(len(boxes))

print(f"\n📊 СТАТИСТИКА ПРЕДСКАЗАНИЙ:")
print(f"  Всего изображений: {len(test_image_names)}")
print(f"  С детекциями staff: {sum(1 for c in staff_counts if c > 0)}")
print(f"  Без staff: {sum(1 for c in staff_counts if c == 0)}")
print(f"  Всего staff: {sum(staff_counts)}")
print(f"  Среднее staff на изображение: {np.mean(staff_counts):.3f}")



print("\n" + "="*70)
print("ЭТАП 6: ФОРМИРОВАНИЕ SUBMISSION")
print("="*70)

def prepare_submission_rows(predictions):
    """Подготовка строк для submission"""
    json_strings = []
    
    for boxes in predictions:
        if boxes:
            # Сортируем по уверенности
            boxes.sort(key=lambda x: x[4], reverse=True)
            # Ограничиваем количество
            boxes = boxes[:40]
            # Конвертируем в JSON
            json_strings.append(json.dumps(boxes, separators=(',', ':')))
        else:
            json_strings.append('[]')
    
    return json_strings

submission_boxes = prepare_submission_rows(ensemble_predictions)

# Создаем финальный DataFrame
final_submission = pd.DataFrame({
    'id': sample_df['id'],
    'image_name': sample_df['image_name'],
    'boxes': submission_boxes
})


output_filename = f'{config.WORK_DIR}/submission_3fold_ensemble.csv'
final_submission.to_csv(output_filename, index=False)

print(f"\n💾 ФАЙЛ СОХРАНЕН: {output_filename}")
print(f"📏 Размер файла: {os.path.getsize(output_filename) / 1024:.2f} KB")


print("\n" + "="*70)
print("ЭТАП 7: ИТОГОВАЯ СТАТИСТИКА")
print("="*70)

print("\n ИНФОРМАЦИЯ О МОДЕЛЯХ:")
for i, model_info in enumerate(trained_models):
    model_size = os.path.getsize(model_info['path']) / (1024*1024) if os.path.exists(model_info['path']) else 0
    print(f"  Модель {i+1} (Fold {model_info['fold']}): {model_size:.2f} MB")

print(f"\n ПАРАМЕТРЫ:")
print(f"  Количество фолдов: {config.N_FOLDS}")
print(f"  Оптимальный порог: {optimal_threshold:.2f}")
print(f"  Устройство: {config.DEVICE}")

if config.DEVICE == 'cuda':
    print(f"\n ПАМЯТЬ GPU:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"  Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    print(f"  Peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

print("\n" + "="*70)
print("✅ ПРОЦЕСС ЗАВЕРШЕН УСПЕШНО")
print("="*70)

# Ссылка для скачивания
from IPython.display import FileLink, display
display(FileLink(output_filename))

✅ SEED УСТАНОВЛЕН: 42
ЭТАП 1: ЗАГРУЗКА И АНАЛИЗ ДАННЫХ
Найдено изображений: 3908


Анализ меток:   0%|          | 0/3908 [00:00<?, ?it/s]


РЕЗУЛЬТАТЫ АНАЛИЗА:
  Изображения с обоими классами:  2662
  Только customer:   819
  Только staff:      427
  Пустые:              0
  Всего боксов:    188502
  Customer боксов: 17066
  Staff боксов:     4060

ЭТАП 2: ПОДГОТОВКА 3-ФОЛД КРОСС-ВАЛИДАЦИИ


Создание страт:   0%|          | 0/3908 [00:00<?, ?it/s]

  Fold 0: train=2605, val=1303
  Fold 1: train=2605, val=1303
  Fold 2: train=2606, val=1302

ЭТАП 3: ОБУЧЕНИЕ 3-х МОДЕЛЕЙ

 ОБРАБОТКА FOLD 0
--------------------------------------------------


...

Порог 0.35:   0%|          | 0/50 [00:00<?, ?it/s]

  Порог 0.35: 161 детекций, rate=0.82

✅ Оптимальный порог: 0.10

ЭТАП 5: ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ


Обработка тестовых изображений:   0%|          | 0/4454 [00:00<?, ?it/s]


📊 СТАТИСТИКА ПРЕДСКАЗАНИЙ:
  Всего изображений: 4454
  С детекциями staff: 1969
  Без staff: 2485
  Всего staff: 2757
  Среднее staff на изображение: 0.619

ЭТАП 6: ФОРМИРОВАНИЕ SUBMISSION

💾 ФАЙЛ СОХРАНЕН: /kaggle/working/submission_3fold_ensemble.csv
📏 Размер файла: 392.79 KB

ЭТАП 7: ИТОГОВАЯ СТАТИСТИКА

 ИНФОРМАЦИЯ О МОДЕЛЯХ:
  Модель 1 (Fold 0): 49.60 MB
  Модель 2 (Fold 1): 49.60 MB
  Модель 3 (Fold 2): 49.60 MB

 ПАРАМЕТРЫ:
  Количество фолдов: 3
  Оптимальный порог: 0.10
  Устройство: cuda

 ПАМЯТЬ GPU:
  Allocated: 0.83 GB
  Cached: 4.93 GB
  Peak: 6.73 GB

✅ ПРОЦЕСС ЗАВЕРШЕН УСПЕШНО


/kaggle/working/submission_3fold_ensemble.csv

In [5]:
display(FileLink('/kaggle/working/yolo_fold_2/run/weights/best.pt'))

/kaggle/working/yolo_fold_0/run/weights/best.pt

In [6]:
# Создание архива всех результатов
import zipfile
import os

with zipfile.ZipFile('/kaggle/working/full_results.zip', 'w') as zipf:
    # Добавляем сабмишн
    zipf.write('/kaggle/working/submission_3fold_ensemble.csv', 'submission.csv')
    
    # Добавляем все обученные модели
    for i in range(3):
        model_path = f'/kaggle/working/yolo_fold_{i}/run/weights/best.pt'
        if os.path.exists(model_path):
            zipf.write(model_path, f'model_fold_{i}.pt')
    
    # Добавляем информацию
    if os.path.exists('/kaggle/working/fold_0_dataset/data.yaml'):
        zipf.write('/kaggle/working/fold_0_dataset/data.yaml', 'data.yaml')

print("✅ Полный архив создан: /kaggle/working/full_results.zip")
display(FileLink('full_results.zip'))

✅ Полный архив создан: /kaggle/working/full_results.zip


/kaggle/working/full_results.zip